In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

I0000 00:00:1790282034.479081    4385 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790282034.811512    4385 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1790282037.145403    4385 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [7]:
import os
Dataset_dir = "/mnt/e/kartik/Mango flower disease detection DL project/mango_dataset"

train_ds = os.path.join(Dataset_dir , "train")
balanced_train_ds = os.path.join(Dataset_dir , "train_balanced")
val_dir = os.path.join(Dataset_dir , "validation") 
test_dir = os.path.join(Dataset_dir , "test")
model_dir = os.path.join(Dataset_dir , "models")

os.makedirs(model_dir , exist_ok = True)

print("Dataset dir :- " , Dataset_dir)
print("Train dir :- " , train_ds)
print("Balanced Train dir :- " , balanced_train_ds)
print("Validation dir :- " , val_dir)
print("Test dir :- " , test_dir)
print("Model dir :- " , model_dir)

Dataset dir :-  /mnt/e/kartik/Mango flower disease detection DL project/mango_dataset
Train dir :-  /mnt/e/kartik/Mango flower disease detection DL project/mango_dataset/train
Balanced Train dir :-  /mnt/e/kartik/Mango flower disease detection DL project/mango_dataset/train_balanced
Validation dir :-  /mnt/e/kartik/Mango flower disease detection DL project/mango_dataset/validation
Test dir :-  /mnt/e/kartik/Mango flower disease detection DL project/mango_dataset/test
Model dir :-  /mnt/e/kartik/Mango flower disease detection DL project/mango_dataset/models


In [36]:
IMG_SIZE = (256 , 256)
BATCH_SIZE = 16
SEED = 42

train_ds = keras.utils.image_dataset_from_directory(
    balanced_train_ds ,
    labels = "inferred" ,
    label_mode = "int" ,
    image_size = IMG_SIZE ,
    batch_size = BATCH_SIZE ,
    shuffle = True ,
    seed = SEED
)

validation_ds = keras.utils.image_dataset_from_directory(
    val_dir ,
    labels = "inferred" ,
    label_mode = "int" , 
    image_size = IMG_SIZE ,
    batch_size = BATCH_SIZE ,
    shuffle = True ,
    seed = SEED
)

test_ds = keras.utils.image_dataset_from_directory(
    test_dir ,
    labels = "inferred" ,
    label_mode = "int" ,
    image_size = IMG_SIZE ,
    batch_size = BATCH_SIZE ,
    shuffle = True ,
    seed = SEED
)

class_names = train_ds.class_names

print("Classes")
for i , name in enumerate(class_names):
    print(i , " " , name)

Found 1400 files belonging to 7 classes.
Found 139 files belonging to 7 classes.
Found 151 files belonging to 7 classes.
Classes
0   anthracnose
1   anthracnose & mango malformation
2   anthracnose & powdery mildew
3   healthy
4   mango malformation
5   mango malformation & powdery mildew
6   powdery mildew


In [37]:
print(train_ds.class_names)

['anthracnose', 'anthracnose & mango malformation', 'anthracnose & powdery mildew', 'healthy', 'mango malformation', 'mango malformation & powdery mildew', 'powdery mildew']


In [38]:
print(len(train_ds.class_names))

7


In [39]:
for images , labels in train_ds.take(1):
    print(images.shape)
    print(labels.shape)

(16, 256, 256, 3)
(16,)


In [40]:
for images , labels in train_ds.take(1):
    print("Image min:" , tf.reduce_min(images).numpy())
    print("Image max:" , tf.reduce_max(images).numpy())

Image min: 0.0
Image max: 255.0


In [41]:
normalisation_layer = layers.Rescaling(1./255)

for images , labels in train_ds.take(1):
    normalised_images = normalisation_layer(images)

    print("Before:", tf.reduce_min(images).numpy(),
          tf.reduce_max(images).numpy())

    print("After:", tf.reduce_min(normalised_images).numpy(),
          tf.reduce_max(normalised_images).numpy())

Before: 0.0 255.0
After: 0.0 1.0


In [42]:
base_model = keras.applications.EfficientNetB0(
    include_top = False,
    weights = "imagenet"
)
base_model.trainable = False

In [43]:
inputs = keras.Input(shape = IMG_SIZE + (3,))
x = base_model(inputs , training = False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dense(128 , activation = "relu")(x)
x = keras.layers.Dropout(0.5)(x)
outputs = keras.layers.Dense(7 , activation = "softmax")(x)
model = keras.Model(inputs , outputs)


In [44]:
model.compile(loss = "sparse_categorical_crossentropy" , optimizer = keras.optimizers.Adam(learning_rate = 0.001) , metrics = ['accuracy'])

In [45]:
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_6 (InputLayer)      │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 8, 8, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,214,442 (16.08 MB)

 Trainable params: 164,871 (644.03 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [46]:
checkpoint_path = os.path.join(
    model_dir,
    "efficientnetv2b0_new.keras"
)

checkpoint = keras.callbacks.ModelCheckpoint(
    checkpoint_path,
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [47]:
history = model.fit(train_ds , 
                    validation_data = validation_ds ,
                    epochs = 20 ,
                    callbacks = [checkpoint , early_stopping])

Epoch 1/20
 5/88 ━━━━━━━━━━━━━━━━━━━━ 26s 318ms/step - accuracy: 0.1750 - loss: 2.0168

W0000 00:00:1790285969.384070   16906 op_kernel.cc:1858] OP_REQUIRES failed at whole_file_read_ops.cc:117 : RESOURCE_EXHAUSTED: /mnt/e/kartik/Mango flower disease detection DL project/mango_dataset/train_balanced/anthracnose/IMG_20260208_113155439_HDR_AE.jpg; Cannot allocate memory


 7/88 ━━━━━━━━━━━━━━━━━━━━ 36s 455ms/step - accuracy: 0.1875 - loss: 1.9570

W0000 00:00:1790285970.264823   16922 op_kernel.cc:1858] OP_REQUIRES failed at whole_file_read_ops.cc:117 : RESOURCE_EXHAUSTED: /mnt/e/kartik/Mango flower disease detection DL project/mango_dataset/train_balanced/powdery mildew/IMG_20260207_144444819_HDR_AE.jpg; Cannot allocate memory


10/88 ━━━━━━━━━━━━━━━━━━━━ 32s 413ms/step - accuracy: 0.2438 - loss: 1.8677

ResourceExhaustedError: Graph execution error:

Detected at node ReadFile defined at (most recent call last):
<stack traces unavailable>
/mnt/e/kartik/Mango flower disease detection DL project/mango_dataset/train_balanced/anthracnose/IMG_20260208_113155439_HDR_AE.jpg; Cannot allocate memory
	 [[{{node ReadFile}}]]
	 [[IteratorGetNext]]
Hint: If you want to see a list of allocated tensors when OOM happens, add report_tensor_allocations_upon_oom to RunOptions for current allocation info. This isn't available when running in Eager mode.
 [Op:__inference_multi_step_on_iterator_33651]